In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "data_quality":
    os.chdir("..")

# Use interactive widgets to define parameters for evaluating Data Quality

In [ ]:
try:
    import ipywidgets
except ImportError as e:
    !pip install ipywidgets

from data_quality.gx.plugins.utils import initialize_widgets
from IPython.display import display

widgets_combined = initialize_widgets()
display(widgets_combined)

- ### Initialize the Spark connection
- ### Read data based on the selected parameters

In [ ]:
from data_quality.gx.plugins.connectors import SparkConnector

connector = SparkConnector()
connector.connect()
dtlk_df = connector.read_data()

## Initialize Great Expectations:

### Setup GX context:

 - _Define GX Data Source:_
   - A Data Source provides a standard API for accessing and interacting with data from a wide variety of source systems.
 - _Define GX Data Asset:_
    - A Data Asset is a collection of records within a Data Source which is usually named based on the underlying data system and sliced specification.
 - Define Batch:
    - A Batch is a selection of records from a Data Asset.
 - Define Expectation Suite:
   - An Expectation Suite is a collection of verifiable assertions about data.
 - Define GX validator:
    - A Validator is the object responsible for running an Expectation Suite against data.

In [ ]:
import great_expectations as gx

context = gx.get_context(project_root_dir="./data_quality")

dq_suite = f"{connector.layer}_{connector.datasource}_suite"
dq_asset = f"{connector.layer}_{connector.datasource}_asset"
dataframe_datasource = context.sources.add_or_update_spark(name=f"{connector.layer}_{connector.datasource}")

dq_asset = dataframe_datasource.add_dataframe_asset(name=dq_asset,dataframe=dtlk_df)

In [ ]:
checkpoint = context.get_checkpoint(f"{connector.layer}_{connector.datasource}_checkpoint")
checkpoint_results = checkpoint.run()

## Generate Custom Data Quality Report


### Completeness checks:
> Availability of required data attributes:
>  - There are no gaps in data structure (all fields are populated)
> - Checks for data gaps, NULL distribution, column shifts and etc
#### GX methods:
 - _expect_column_to_exist()_ - Expect the specified column to exist.
 - _expect_column_values_to_not_be_null()_ - Expect the column values to not be null.
 - _expect_table_row_count_to_be_between()_ - Expect the number of rows to be between two values. Possible to add __max__ value as well.
 - _expect_table_column_count_to_equal()_ - Expect the number of columns in a table to equal a value. Method counts not only high-level but also every nested column in _STRUCT_ attributes.


### Timeliness checks:
> The currency of content representation as well as whether the data is available/can be used when needed:
> - Data is available upon request and when required.
> - Measures are established to track and when possible, optimize data load times, report generation times, interface response times.
#### GX methods:
 - _expect_column_max_to_be_between()_ - Expect the column maximum to be between a minimum value and a maximum value.


### Validity checks:
> Alignment of data content with required standards:
> - Values conform to pre-defined ranges.
> - No format mismatches.
> - No issues with casting data types when loading.
#### GX methods:
- _expect_column_values_to_match_regex()_ - Expect the column entries to be strings that match a given regular expression.
- _expect_column_values_to_be_of_type()_ - Expect a column to contain values of a specified data type.
- _expect_column_values_to_be_between()_ - Expect the column value to be between a minimum value and a maximum value.
- _expect_column_values_to_be_json_parseable()_ - Expect the column entries to be data written in Json (JavaScript Object Notation


### Integrity checks:
> How well the data complies with the required formats/definitions:
> - The structure of relationships within the data is correctly maintained
> - Keys and relationships are properly established and preserved, enabling complex queries across multiple datasets
#### GX methods:
- _expect_column_values_to_be_in_set()_ - Ensures that each value in a column belongs to a predefined set of valid value(s)


### Uniqueness checks:
> Focus on garanting the uniquenness of values or combinations of values within data entity:
> - Uniqueness constraint violations of primary key(s)
> - Uniqueness for columns (or their combinations) that contain business-critical or domain-specific logic
#### GX methods:
- _expect_compound_columns_to_be_unique()_ - Expect the compound columns to be unique.
- _expect_column_values_to_be_unique()_ - Expect each column value to be unique. This expectation detects duplicates. All duplicated values are counted as exceptions.

In [ ]:
connector.generate_report(checkpoint_results, connector.mode)
connector.show_report(connector.layer, connector.datasource, detailed_report=True)
connector.save_report()